In [ ]:
import os
import sys
import glob

import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs
from utils.hampel_filter import apply_hampel_filter

In [2]:
csv_files = Configs.ICATER_FILES

df = pd.concat([pd.read_csv(f, delimiter=",") for f in csv_files], ignore_index=True)

# Map movement types to labels
mapa_mov = {0: 'Pick', 1: 'Place', 6: 'Move'} 
df['movLabel'] = df['movType'].map(mapa_mov)

# Add Jerk column
df["jerk"] = df.groupby(['sub', 'repID', 'movID'])['acc'].diff().fillna(0)

df.head()

,sub,repID,movID,movType,step,xs,ys,zs,xe,ye,ze,xw,yw,zw,vel,acc,t,k,movLabel,jerk
0,6,1,1,0,1,-19.429,191.439,1455.13,-73.750,205.827,1143.18,-187.531407,217.476491,910.602189,0.012873,0.021275,-0.172269,2.697837,Pick,0.000000
1,6,1,1,0,2,-19.497,191.474,1455.08,-73.891,205.878,1143.14,-187.744348,217.450322,910.600828,0.012939,0.039943,-0.465353,4.939217,Pick,0.018667
2,6,1,1,0,3,-19.542,191.511,1455.03,-74.009,205.962,1143.11,-187.960732,217.435456,910.599790,0.013292,0.074186,-1.444924,8.564583,Pick,0.034243
3,6,1,1,0,4,-19.555,191.559,1454.98,-74.121,206.100,1143.07,-188.187275,217.438896,910.600447,0.014188,0.103099,-3.299226,10.266306,Pick,0.028913
4,6,1,1,0,5,-19.533,191.617,1454.90,-74.220,206.297,1143.03,-188.432741,217.464603,910.604255,0.015738,0.125656,-6.755100,9.310180,Pick,0.022558


## Filtragem

### Filtro de Hampel (ou Mediana) para tentar diminuir os erros do sensor

In [3]:
# (Ombro: xs, ys, zs | Cotovelo: xe, ye, ze | Mão: xw, yw, zw)
sensor_cols = ['xs', 'ys', 'zs', 'xe', 'ye', 'ze', 'xw', 'yw', 'zw']

for col in sensor_cols:
    print(f"Processing column with Hampel: {col} ...")

    df[col + '_clean'] = df.groupby(['sub', 'repID', 'movID'])[col].transform(
        lambda x: apply_hampel_filter(x, window_size=5)
    )

print("Done")

Processing column with Hampel: xs ...
Processing column with Hampel: ys ...
Processing column with Hampel: zs ...
Processing column with Hampel: xe ...
Processing column with Hampel: ye ...
Processing column with Hampel: ze ...
Processing column with Hampel: xw ...
Processing column with Hampel: yw ...
Processing column with Hampel: zw ...
Done


### Filtro de Savitzky-Golay para suavizar o resultado final 

In [4]:
# APPLYING SAVITZKY-GOLAY FILTER TO SMOOTH THE DATA. 
# This will help reduce noise in the trajectory data, 
# will make things cleaner for visualization (or outlier detection).

window = 11  
order = 3    

def apply_savgol(series):
    return savgol_filter(series, window_length=window, polyorder=order)

cols_to_filter = [f"{col}_clean" for col in sensor_cols]
for col in cols_to_filter:
    print(f"Processing column with Savitzky-Golay: {col} ...")
    df[f'{col}_smooth'] = df.groupby(['sub', 'repID', 'movID'])[col].transform(apply_savgol)

Processing column with Savitzky-Golay: xs_clean ...
Processing column with Savitzky-Golay: ys_clean ...
Processing column with Savitzky-Golay: zs_clean ...
Processing column with Savitzky-Golay: xe_clean ...
Processing column with Savitzky-Golay: ye_clean ...
Processing column with Savitzky-Golay: ze_clean ...
Processing column with Savitzky-Golay: xw_clean ...
Processing column with Savitzky-Golay: yw_clean ...
Processing column with Savitzky-Golay: zw_clean ...


### Clean Final Dataframe

In [5]:
# Store the metadata and the new 9 smoothed columns + derived kinematic features
metadata_cols = ['sub', 'repID', 'movID', 'movType', 'step', 'movLabel']
smooth_cols = [f"{col}_clean_smooth" for col in sensor_cols]
kinematic_cols = ['vel', 'acc', 't', 'k', 'jerk']

df = df[metadata_cols + smooth_cols + kinematic_cols]

# Rename the smoothed columns to their original names for consistency in DTW
df.columns = metadata_cols + sensor_cols + kinematic_cols

### Save to Csv

In [6]:
cache_path = Configs.ICATER_ALL_SENSORS_FILTERED_DATA_DIR
df.to_csv(cache_path, index=False)
print(f"Processed data saved to {cache_path}")

Processed data saved to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/ICater/data/processed/all_sensors_filtered_data.csv
